# Imports

In [1]:
import torch
import sys
import os
from torch import Tensor
sys.path.append('..')
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from definitions import EXPERIMENTS_DIR, ROOT_DIR
from src.commands.train import Trainer, Config
from torch_geometric.data import HeteroData
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
)
from PIL import ImageColor

In [2]:
px.defaults.template = 'seaborn'

In [3]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Load the results

In [4]:
# TASK = 'hierarchical_classification'
EXPERIMENT = 'nantes-uni-2020_e5-v3'
CORPUS = 'nantes-uni-2020'
EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, 'results')
HISTORIES_DIR = os.path.join(EXPERIMENT_DIR, 'histories')
CONFIG_PATH = os.path.join(EXPERIMENT_DIR, 'config.json')
SPLIT_PATH = os.path.join(EXPERIMENT_DIR, 'split_sub_dsis_4.pt')

In [5]:
history = pd.concat([
    pd.read_csv(os.path.join(HISTORIES_DIR, filename))
    for filename in os.listdir(HISTORIES_DIR)
])
history = history \
    .groupby(['epoch','phase','config','trial']) \
    .mean() \
    .reset_index()
history.head()

,epoch,phase,config,trial,subdomain_guidance_loss,lab_guidance_loss,loss,subdomain_guidance_acc,subdomain_guidance_f1,lab_guidance_acc,lab_guidance_f1
0,0,train,Conv_Attn_null_False,0,0.182027,0.114998,0.297025,0.929270,0.314434,0.956299,0.369033
1,0,train,Conv_Attn_null_False,1,0.181132,0.110035,0.291167,0.930641,0.303813,0.958474,0.382964
2,0,train,Conv_Attn_null_False,2,0.182429,0.108238,0.290667,0.929212,0.300950,0.960625,0.392052
3,0,train,Conv_Attn_null_False,3,0.182341,0.105477,0.287819,0.929155,0.302341,0.962028,0.402359
4,0,train,Conv_Max_null_False,0,0.192907,0.123437,0.316344,0.925797,0.282015,0.953891,0.349739


In [6]:
history_mean = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).mean() \
    .add_suffix('_mean')

history_std = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).std() \
    .add_suffix('_std')

In [7]:
history = history_mean.join(other = history_std).reset_index()
history.head()

,epoch,phase,config,subdomain_guidance_loss_mean,lab_guidance_loss_mean,loss_mean,subdomain_guidance_acc_mean,subdomain_guidance_f1_mean,lab_guidance_acc_mean,lab_guidance_f1_mean,subdomain_guidance_loss_std,lab_guidance_loss_std,loss_std,subdomain_guidance_acc_std,subdomain_guidance_f1_std,lab_guidance_acc_std,lab_guidance_f1_std
0,0,train,Conv_Attn_null_False,0.181982,0.109687,0.291669,0.929570,0.305385,0.959356,0.386602,0.000593,0.004006,0.003863,0.000716,0.006145,0.002508,0.014141
1,0,train,Conv_Max_null_False,0.191311,0.120485,0.311795,0.926172,0.276494,0.955153,0.353301,0.001481,0.002639,0.003175,0.000799,0.006053,0.001294,0.006893
2,0,train,Conv_Mean_null_False,0.182586,0.109690,0.292276,0.929198,0.303606,0.959666,0.387353,0.002953,0.001460,0.003254,0.001075,0.003649,0.000622,0.005611
3,0,val,Conv_Attn_null_False,0.190172,0.091402,0.281574,0.935679,0.337951,0.972959,0.449628,0.011955,0.002855,0.011783,0.004145,0.022654,0.001547,0.011714
4,0,val,Conv_Max_null_False,0.196808,0.084285,0.281093,0.931582,0.319630,0.973548,0.441821,0.018311,0.004835,0.013709,0.004709,0.013424,0.000803,0.012985


# Load Data

In [8]:
config = Config(experiment=EXPERIMENT)
trainer = Trainer(experiment=config.experiment, config=config.training_config)
configs = trainer.get_configs()

In [9]:
data = trainer.load_data()
# data = trainer.filter_data(data)
# data = trainer.keep_largest_cc(data) if trainer.config.keep_largest_cc else data
data.split = trainer.create_or_load_split(data)
data['Article'].split = data.split

In [10]:
metadata = data.raw_metadata

# Learning Curves

In [11]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> None:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['config'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and config == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

In [12]:
line_plot(history, 'loss')

In [13]:
line_plot(history, 'subdomain_guidance_acc')

In [14]:
line_plot(history, 'subdomain_guidance_f1')

In [15]:
line_plot(history, 'lab_guidance_acc')

In [16]:
line_plot(history, 'lab_guidance_f1')

# Calculate other metrics

## Load Predictions

In [17]:
def load_results(results_dir : str) -> dict[tuple[str, str], Tensor]:

    results = dict()

    for filename in os.listdir(results_dir):

        parts = filename.split('.')[0].split('_')
        trial = parts[-1]
        expr = '_'.join(parts[1:-1])

        results[(expr,trial)] = torch.load(os.path.join(RESULTS_DIR, filename), map_location='cpu', weights_only=False)

    return results

In [18]:
results = load_results(RESULTS_DIR)

## Label Wise & Row Wise Metrics

In [19]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [20]:
def get_experiment_params(expr : str) -> dict:

    parts = expr.split('_')
        
    return {
        "GNN Variant" : parts[0],
        "Readout Function" : parts[1],
        "Pooling Layer" : parts[2],
        "Undersampling" : parts[3]
    }

In [21]:
def get_label_wise_metrics(
    label_name : str,
    data : HeteroData,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    n_labels = data[f'y_{label_name}'] if label_name != 'lab' else data[f'y_{label_name}'][:,:-1]
    n_labels = n_labels.size(-1)

    for label in range(n_labels):
            
        for (expr,trial),result in results.items():

            y = data[f'y_{label_name}'][data.split == int(trial[-1])]
            y_hat = result[label_name][data.split == int(trial[-1])].sigmoid()
            y_hard = y_hat.gt(0.5).long()
            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label]),
                "precision" : precision_score(y[:,label], y_hard[:,label]),
                "recall" : recall_score(y[:,label], y_hard[:,label]),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label])
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [22]:
def get_row_wise_metrics(
    label : str,
    data : HeteroData,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        y = data[f'y_{label}'][data.split == int(trial[-1])]
        y_hat = result[label][data.split == int(trial[-1])].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        if label == 'lab':
            y = y[:,:-1]
            y_hat = y_hat[:,:-1]
            y_hard = y_hard[:,:-1]

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro'),
            "precision" : precision_score(y, y_hard , average='macro'),
            "recall" : recall_score(y, y_hard , average='macro'),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [23]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('subdomain', data['Article'], results, metadata['labels']['subdomain'])
display_mean_std_df(label_wise_metrics_agg)

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined

,expr,recall,f1_score,precision,accuracy,auc
0,full_Conv_Attn_null_False,69.88% ± 1.80%,73.75% ± 1.74%,78.95% ± 2.21%,95.76% ± 0.27%,96.91% ± 0.17%
1,full_Conv_Max_null_False,68.81% ± 2.13%,72.95% ± 1.79%,78.85% ± 2.47%,95.56% ± 0.35%,96.60% ± 0.42%
2,full_Conv_Mean_null_False,66.69% ± 2.04%,72.21% ± 1.62%,80.44% ± 1.72%,95.75% ± 0.23%,96.97% ± 0.10%
3,partial_Conv_Attn_null_False,16.33% ± 3.58%,15.11% ± 1.89%,59.69% ± 12.32%,86.09% ± 1.47%,63.83% ± 2.90%
4,partial_Conv_Max_null_False,26.79% ± 12.87%,30.66% ± 15.81%,66.57% ± 4.95%,90.00% ± 2.20%,80.55% ± 8.02%
5,partial_Conv_Mean_null_False,21.67% ± 2.30%,27.49% ± 1.86%,81.23% ± 4.97%,91.89% ± 0.67%,91.53% ± 0.36%


In [24]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics('subdomain', data['Article'], results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:



,expr,recall,f1_score,precision,accuracy,hamming_loss,auc
0,full_Conv_Attn_null_False,69.88% ± 1.80%,73.75% ± 1.74%,78.95% ± 2.21%,52.58% ± 1.43%,4.24% ± 0.27%,96.91% ± 0.17%
1,full_Conv_Max_null_False,68.81% ± 2.13%,72.95% ± 1.79%,78.85% ± 2.47%,50.89% ± 2.24%,4.44% ± 0.35%,96.60% ± 0.42%
2,full_Conv_Mean_null_False,66.69% ± 2.04%,72.21% ± 1.62%,80.44% ± 1.72%,52.91% ± 1.17%,4.25% ± 0.23%,96.97% ± 0.10%
3,partial_Conv_Attn_null_False,16.33% ± 3.58%,15.11% ± 1.89%,59.69% ± 12.32%,6.77% ± 1.33%,13.91% ± 1.47%,63.83% ± 2.90%
4,partial_Conv_Max_null_False,26.79% ± 12.87%,30.66% ± 15.81%,66.57% ± 4.95%,21.96% ± 11.20%,10.00% ± 2.20%,80.55% ± 8.02%
5,partial_Conv_Mean_null_False,21.67% ± 2.30%,27.49% ± 1.86%,81.23% ± 4.97%,32.91% ± 2.69%,8.11% ± 0.67%,91.53% ± 0.36%


In [26]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('lab', data['Article'], results, metadata['labels']['lab'])
display_mean_std_df(label_wise_metrics_agg)

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined

,expr,recall,f1_score,precision,accuracy,auc
0,full_Conv_Attn_null_False,82.67% ± 3.83%,85.87% ± 3.01%,90.36% ± 1.88%,98.13% ± 0.32%,98.80% ± 0.46%
1,full_Conv_Max_null_False,80.75% ± 7.72%,83.87% ± 6.21%,90.08% ± 3.06%,97.79% ± 0.39%,98.52% ± 0.42%
2,full_Conv_Mean_null_False,80.94% ± 5.00%,85.11% ± 3.99%,91.55% ± 2.27%,98.00% ± 0.36%,98.89% ± 0.23%
3,partial_Conv_Attn_null_False,4.79% ± 1.31%,8.61% ± 2.15%,64.99% ± 11.61%,90.39% ± 0.22%,62.72% ± 3.67%
4,partial_Conv_Max_null_False,19.49% ± 16.39%,25.64% ± 16.83%,69.93% ± 6.48%,92.46% ± 1.66%,81.86% ± 10.09%
5,partial_Conv_Mean_null_False,24.56% ± 1.76%,32.58% ± 2.00%,76.47% ± 8.24%,93.83% ± 0.17%,94.46% ± 0.40%


In [27]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics('lab', data['Article'], results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:



,expr,recall,f1_score,precision,accuracy,hamming_loss,auc
0,full_Conv_Attn_null_False,82.67% ± 3.83%,85.87% ± 3.01%,90.36% ± 1.88%,87.22% ± 2.18%,1.87% ± 0.32%,98.80% ± 0.46%
1,full_Conv_Max_null_False,80.75% ± 7.72%,83.87% ± 6.21%,90.08% ± 3.06%,84.87% ± 2.78%,2.21% ± 0.39%,98.52% ± 0.42%
2,full_Conv_Mean_null_False,80.94% ± 5.00%,85.11% ± 3.99%,91.55% ± 2.27%,86.28% ± 2.10%,2.00% ± 0.36%,98.89% ± 0.23%
3,partial_Conv_Attn_null_False,4.79% ± 1.31%,8.61% ± 2.15%,64.99% ± 11.61%,7.14% ± 2.58%,9.61% ± 0.22%,62.72% ± 3.67%
4,partial_Conv_Max_null_False,19.49% ± 16.39%,25.64% ± 16.83%,69.93% ± 6.48%,32.56% ± 20.71%,7.54% ± 1.66%,81.86% ± 10.09%
5,partial_Conv_Mean_null_False,24.56% ± 1.76%,32.58% ± 2.00%,76.47% ± 8.24%,45.96% ± 0.41%,6.17% ± 0.17%,94.46% ± 0.40%


In [27]:
hparams = metrics.apply(lambda x : pd.Series(get_experiment_params(x['expr'])), axis = 1)
metrics = hparams.join(metrics, lsuffix="hparams")
metrics['Experiment'] = metrics['expr']
metrics = metrics.drop(columns = ['expr'])
metrics = metrics[[metrics.columns[-1], *metrics.columns[:-1]]]
# metrics.to_csv(os.path.join(dir, "hierarchical_teams_metrics.xlsx"), index = False)
metrics

,Experiment,GNN Variant,Readout Function,Pooling Layer,Undersampling,f1_score,auc,hamming_loss,accuracy,recall,precision
0,full_Conv_Attn_null_False,full,Conv,Attn,null,85.87% ± 3.01%,98.80% ± 0.46%,1.87% ± 0.32%,87.22% ± 2.18%,82.67% ± 3.83%,90.36% ± 1.88%
1,full_Conv_Max_null_False,full,Conv,Max,null,83.87% ± 6.21%,98.52% ± 0.42%,2.21% ± 0.39%,84.87% ± 2.78%,80.75% ± 7.72%,90.08% ± 3.06%
2,full_Conv_Mean_null_False,full,Conv,Mean,null,85.11% ± 3.99%,98.89% ± 0.23%,2.00% ± 0.36%,86.28% ± 2.10%,80.94% ± 5.00%,91.55% ± 2.27%
3,partial_Conv_Attn_null_False,partial,Conv,Attn,null,8.61% ± 2.15%,62.72% ± 3.67%,9.61% ± 0.22%,7.14% ± 2.58%,4.79% ± 1.31%,64.99% ± 11.61%
4,partial_Conv_Max_null_False,partial,Conv,Max,null,25.64% ± 16.83%,81.86% ± 10.09%,7.54% ± 1.66%,32.56% ± 20.71%,19.49% ± 16.39%,69.93% ± 6.48%
5,partial_Conv_Mean_null_False,partial,Conv,Mean,null,32.58% ± 2.00%,94.46% ± 0.40%,6.17% ± 0.17%,45.96% ± 0.41%,24.56% ± 1.76%,76.47% ± 8.24%


- Visualizations

In [28]:
def bar_plot(df : pd.DataFrame, metric : str) -> None:

    fig = px.bar(
        data_frame = df.sort_values(by = f'{metric}_mean'),
        x = 'expr',
        y = f'{metric}_mean',
        barmode = 'group',
        error_y = f'{metric}_std',
        text = f'{metric}_mean',
        width = 950,
        height = 800
    )

    fig.update_layout(
        xaxis_title = 'Experiment',
        yaxis_title = metric.capitalize().replace('_',' '),
    )

    fig.update_traces(
        textposition = 'outside',
        texttemplate = '%{text:.2%}'
    )

    fig.show()

In [29]:
bar_plot(label_wise_metrics_agg, 'accuracy')

In [30]:
bar_plot(label_wise_metrics_agg, 'f1_score')

In [31]:
bar_plot(row_wise_metrics_agg, 'accuracy')